# GenRec Hint Cascade 级联分析

这份 notebook 用来阅读 `analyze_rl_beam_hint.py` 生成的 `summary.json` / `details.json` 结果，并把级联 hint 分析翻译成更容易解释的中文结论。

这份分析主要回答四个问题：
- 不给任何真值提示时，`beam search` 本身能命中多少样本？
- 给出前 `1 / 2 / 3` 个真值 SID token 作为提示后，剩余困难样本还能追回多少？
- 命中率提升来自“模型真的会续写”，还是只是因为提示直接送掉了前缀？
- 到最后还剩下哪些顽固难例，值得单独排查？

如果你没有跑过这个脚本，也可以直接把这份 notebook 当成“结果解释器”来看。需要注意的是：如果当前环境没有对应的 `summary/details` 缓存，图表与表格需要你自己重新运行单元格后才能刷新。


## `analyze_rl_beam_hint.py` 在做什么

脚本的核心流程是一个“失败样本逐级重试”的 cascade：

1. 在 `base` 阶段，不给任何真值 SID 前缀，直接对每个样本做 beam search。
2. 对每个样本得到的一组 beam 候选，同时计算两类信号：
   - `exact hit / rule hit`：是否至少有一条候选与真值 SID 序列完全一致。
   - `prefix match`：每条候选从左到右，连续命中了多少个真值 token。
3. 把 `base` 里没有命中的样本组成困难子集，只对这些样本继续跑 `hint_1`。
4. `hint_1` 会把真值的第 1 个 SID token 当作提示前缀接到 prompt 后面，再看剩余部分能否接对。
5. 同理，把 `hint_1` 仍然失败的样本送去 `hint_2`，再送去 `hint_3`，形成 `base -> hint_1 -> hint_2 -> hint_3` 的级联。
6. 最后输出：
   - `summary.json`：按 stage 聚合后的总体统计。
   - `details.json`：每个样本、每个阶段的一组 beam 细节。

因此，这里的 `stage` 不是训练阶段，而是“在给定 hint 深度下，对上一轮失败样本再尝试一次”的分析阶段。


## 指标词典与读图指南

先记住下面这些概念，后面的表和图就比较容易看：

- `beam=8 / 16`：每个样本保留 8 或 16 条 beam 候选。beam 越大，理论上“撞中正确序列”的机会越高。
- `hint_depth`：提示了多少个真值 SID token。`base=0`，`hint_1=1`，`hint_2=2`，依此类推。
- `exact hit / rule hit`：一组 beam 候选里，只要有 1 条与真值完全一致，就算该样本在这一 stage 命中。
- `stage_rule_hit_sample_rate_within_input`：当前 stage 在自己的输入子集里追回了多少比例的样本。看这个值，就是看“这一层 hint 有多有用”。
- `cumulative_rule_hit_sample_rate`：从 `base` 一直累计到当前 stage，全部样本里已经解决了多少。看这个值，就是看“cascade 到这里总共救回了多少”。
- `prefix match length`：单条候选从第一个 token 开始，连续命中真值的长度；一旦某个位置错了，就停止计数。
- `prefix reward`：`prefix_match_length / ground_truth_length`，范围在 `[0, 1]`。它不是最终命中率，而是“至少前面对了多长”。
- `full`：直接统计完整候选的前缀命中长度。对 `hint_k` 来说，提示本身也会被算进去。
- `suffix-only`：先减去 `hint_depth`，只看提示之后还能继续接对多少。这更接近“真实的续写能力”。
- `rollout-group pattern`：把同一个样本的一组 beam 前缀命中长度压缩成模式串。例如：
  - `0:8`：8 条 beam 全都从第一个待预测 token 开始就错了。
  - `0:4|1:4`：4 条完全没对上，另 4 条只对了 1 个 token。
  - `3:16`：16 条 beam 都至少把前 3 个 token 接对了。
- `all-zero group rate`：pattern 恰好等于 `0:beam_size` 的样本占比。这个值高，通常说明模型在该 stage 上经常“起步就错”。

推荐的阅读顺序是：先看“核心总表”和两张折线图，再看 `full` / `suffix-only` 的直方图对比，最后看 rollout-group pattern 和最终残留难例。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.style.use('default')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


In [ ]:
# 配置区：如果你想固定读取某一组结果文件，可以直接填写路径。
PINNED_SUMMARY_PATH = None
PINNED_DETAILS_PATH = None

CANDIDATE_ROOTS = [
    Path('temp/rl_beam_hint'),
    Path('/mnt/dolphinfs/hdd_pool/docker/user/hadoop-hmart-poistar/fanghaotian/GenRec/temp/rl_beam_hint'),
]
SUMMARY_PATTERNS = ['*cascade*_summary.json', '*beam_hint*_summary.json', '*_summary.json']


def find_latest_file(patterns, roots):
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            candidates.extend(root.glob(pattern))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


summary_path = Path(PINNED_SUMMARY_PATH) if PINNED_SUMMARY_PATH else find_latest_file(SUMMARY_PATTERNS, CANDIDATE_ROOTS)
details_path = Path(PINNED_DETAILS_PATH) if PINNED_DETAILS_PATH else (
    summary_path.with_name(summary_path.name.replace('_summary.json', '_details.json')) if summary_path else None
)

print('summary 文件 =', summary_path)
print('details 文件 =', details_path)


In [ ]:
if summary_path is None or not summary_path.exists():
    raise FileNotFoundError(
        '未找到 cascade summary JSON。请设置 PINNED_SUMMARY_PATH，或把结果文件放到 temp/rl_beam_hint 下。'
    )

summary = json.loads(summary_path.read_text(encoding='utf-8'))
details = json.loads(details_path.read_text(encoding='utf-8')) if details_path and details_path.exists() else None

results = {int(beam_size): payload for beam_size, payload in summary['results'].items()}
beam_sizes = sorted(results)

meta = pd.Series(
    {
        '模型路径': summary.get('model_path'),
        '数据目录': summary.get('data_dir'),
        '索引路径': summary.get('index_path'),
        '样本数': summary.get('num_samples'),
        'Beam 宽度列表': summary.get('beam_sizes'),
        '最大 hint 深度': summary.get('max_hint_depth'),
        '是否复用缓存': summary.get('cache', {}).get('reused'),
        '缓存 details 路径': summary.get('cache', {}).get('details_path'),
        '缓存 summary 路径': summary.get('cache', {}).get('summary_path'),
    }
)

meta


In [ ]:
def stage_sort_key(stage_name: str):
    if stage_name == 'base':
        return (0, 0)
    return (1, int(stage_name.split('_')[-1]))


def stage_display_name(stage_name: str) -> str:
    if stage_name == 'base':
        return 'base（无提示）'
    hint_depth = int(stage_name.split('_')[-1])
    return f'hint_{hint_depth}（给定前 {hint_depth} 个真值 SID）'


def stage_plot_label(stage_name: str) -> str:
    if stage_name == 'base':
        return 'base (no hint)'
    hint_depth = int(stage_name.split('_')[-1])
    return f'hint_{hint_depth} ({hint_depth}-token hint)'


def normalize_int_key_dict(raw):
    raw = raw or {}
    return {int(key): int(value) for key, value in raw.items()}


def all_zero_pattern_count(stage_payload: dict, beam_size: int, *, suffix: bool) -> int:
    key = 'group_pattern_counts_suffix' if suffix else 'group_pattern_counts_full'
    pattern_counts = stage_payload.get(key, {}) or {}
    # `0:beam_size` 表示这一组 beam 全部从第一个待预测 token 开始就错了。
    return int(pattern_counts.get(f'0:{beam_size}', 0))


def format_percent_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    display_frame = frame.copy()
    for column in columns:
        if column in display_frame.columns:
            display_frame[column] = display_frame[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else ''
            )
    return display_frame


def build_stage_frame(summary_dict):
    rows = []
    for beam_size, beam_payload in sorted(((int(k), v) for k, v in summary_dict['results'].items()), key=lambda x: x[0]):
        for stage_name, stage_payload in sorted(beam_payload['stages'].items(), key=lambda item: stage_sort_key(item[0])):
            rows.append(
                {
                    'beam_size': beam_size,
                    'stage_name': stage_name,
                    'stage_label': stage_display_name(stage_name),
                    'stage_plot_label': stage_plot_label(stage_name),
                    'hint_depth': int(stage_payload['hint_depth']),
                    'input_subset_size': int(stage_payload['input_subset_size']),
                    'evaluated_subset_size': int(stage_payload['evaluated_subset_size']),
                    'stage_rule_hit_sample_count': int(stage_payload['stage_rule_hit_sample_count']),
                    'stage_rule_hit_sample_rate_within_input': float(stage_payload['stage_rule_hit_sample_rate_within_input']),
                    'stage_rule_hit_total_count': int(stage_payload['stage_rule_hit_total_count']),
                    'remaining_subset_size': int(stage_payload['remaining_subset_size']),
                    'remaining_subset_rate_within_input': float(stage_payload['remaining_subset_rate_within_input']),
                    'cumulative_rule_hit_sample_count': int(stage_payload['cumulative_rule_hit_sample_count']),
                    'cumulative_rule_hit_sample_rate': float(stage_payload['cumulative_rule_hit_sample_rate']),
                    'full_all_zero_group_count': all_zero_pattern_count(stage_payload, beam_size, suffix=False),
                    'suffix_all_zero_group_count': all_zero_pattern_count(stage_payload, beam_size, suffix=True),
                    'full_all_zero_group_rate': all_zero_pattern_count(stage_payload, beam_size, suffix=False) / max(int(stage_payload['evaluated_subset_size']), 1),
                    'suffix_all_zero_group_rate': all_zero_pattern_count(stage_payload, beam_size, suffix=True) / max(int(stage_payload['evaluated_subset_size']), 1),
                }
            )
    frame = pd.DataFrame(rows)
    frame['stage_order'] = frame['stage_name'].map(stage_sort_key)
    return frame.sort_values(['beam_size', 'stage_order']).reset_index(drop=True)


def build_hist_frame(summary_dict, *, suffix: bool):
    hist_key = 'suffix_prefix_match_hist_all_beams' if suffix else 'full_prefix_match_hist_all_beams'
    rows = []
    for beam_size, beam_payload in sorted(((int(k), v) for k, v in summary_dict['results'].items()), key=lambda x: x[0]):
        for stage_name, stage_payload in sorted(beam_payload['stages'].items(), key=lambda item: stage_sort_key(item[0])):
            hist = normalize_int_key_dict(stage_payload.get(hist_key))
            total = sum(hist.values())
            for match_len, count in hist.items():
                rows.append(
                    {
                        'beam_size': beam_size,
                        'stage_name': stage_name,
                        'stage_label': stage_display_name(stage_name),
                        'stage_plot_label': stage_plot_label(stage_name),
                        'match_len': match_len,
                        'count': count,
                        'share': count / max(total, 1),
                    }
                )
    return pd.DataFrame(rows)


def build_pattern_frame(summary_dict, *, suffix: bool):
    key = 'group_pattern_counts_suffix' if suffix else 'group_pattern_counts_full'
    rows = []
    for beam_size, beam_payload in sorted(((int(k), v) for k, v in summary_dict['results'].items()), key=lambda x: x[0]):
        for stage_name, stage_payload in sorted(beam_payload['stages'].items(), key=lambda item: stage_sort_key(item[0])):
            for pattern, count in (stage_payload.get(key) or {}).items():
                rows.append(
                    {
                        'beam_size': beam_size,
                        'stage_name': stage_name,
                        'stage_label': stage_display_name(stage_name),
                        'stage_plot_label': stage_plot_label(stage_name),
                        'pattern': pattern,
                        'count': int(count),
                    }
                )
    return pd.DataFrame(rows)


stage_df = build_stage_frame(summary)
full_hist_df = build_hist_frame(summary, suffix=False)
suffix_hist_df = build_hist_frame(summary, suffix=True)
full_pattern_df = build_pattern_frame(summary, suffix=False)
suffix_pattern_df = build_pattern_frame(summary, suffix=True)

stage_preview = stage_df[
    [
        'beam_size',
        'stage_name',
        'stage_label',
        'hint_depth',
        'input_subset_size',
        'evaluated_subset_size',
        'stage_rule_hit_sample_count',
        'cumulative_rule_hit_sample_rate',
        'remaining_subset_size',
    ]
].rename(
    columns={
        'beam_size': 'Beam 宽度',
        'stage_name': '阶段代码',
        'stage_label': '阶段解释',
        'hint_depth': '提示深度',
        'input_subset_size': '本阶段输入样本数',
        'evaluated_subset_size': '实际评估样本数',
        'stage_rule_hit_sample_count': '本阶段命中样本数',
        'cumulative_rule_hit_sample_rate': '累计精确命中率',
        'remaining_subset_size': '剩余困难样本数',
    }
)

display(Markdown('## 阶段级数据预览\n下面这张表把 `summary.json` 展开成逐阶段数据，后面的图表都会基于这些字段构造。'))
display(format_percent_columns(stage_preview, ['累计精确命中率']))


## 核心总表

这张表建议按下面的顺序读：

1. 先看“本阶段追回率”，判断当前这一层 hint 是否真的有帮助。
2. 再看“累计精确命中率”，判断 cascade 到这里总共解决了多少样本。
3. 再看“剩余困难样本数”，观察困难子集是否快速收缩。
4. 最后看 `all-zero` 占比：这个值越高，说明该 stage 里越多样本是“所有 beam 一起起步就错”。

其中：
- `Full 全零组占比` 会把提示 token 也算进来，更适合看 `base` 或整体模式。
- `Suffix-only 全零组占比` 会把提示部分扣掉，更适合看 hint 阶段真正的续写难度。


In [ ]:
summary_table = stage_df[
    [
        'beam_size',
        'stage_name',
        'stage_label',
        'hint_depth',
        'input_subset_size',
        'stage_rule_hit_sample_count',
        'stage_rule_hit_sample_rate_within_input',
        'cumulative_rule_hit_sample_rate',
        'remaining_subset_size',
        'full_all_zero_group_rate',
        'suffix_all_zero_group_rate',
    ]
].copy()

summary_table = summary_table.rename(
    columns={
        'beam_size': 'Beam 宽度',
        'stage_name': '阶段代码',
        'stage_label': '阶段解释',
        'hint_depth': '提示深度',
        'input_subset_size': '本阶段输入样本数',
        'stage_rule_hit_sample_count': '本阶段追回样本数',
        'stage_rule_hit_sample_rate_within_input': '本阶段追回率',
        'cumulative_rule_hit_sample_rate': '累计精确命中率',
        'remaining_subset_size': '剩余困难样本数',
        'full_all_zero_group_rate': 'Full 全零组占比',
        'suffix_all_zero_group_rate': 'Suffix-only 全零组占比',
    }
)

display(
    format_percent_columns(
        summary_table,
        ['本阶段追回率', '累计精确命中率', 'Full 全零组占比', 'Suffix-only 全零组占比'],
    )
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for beam_size in beam_sizes:
    beam_frame = stage_df[stage_df['beam_size'] == beam_size].copy()
    axes[0].plot(beam_frame['stage_plot_label'], beam_frame['cumulative_rule_hit_sample_rate'], marker='o', label=f'beam={beam_size}')
    axes[1].plot(beam_frame['stage_plot_label'], beam_frame['stage_rule_hit_sample_rate_within_input'], marker='o', label=f'beam={beam_size}')

axes[0].set_title('Cumulative Exact-Hit Rate by Stage')
axes[0].set_ylabel('Cumulative Sample Hit Rate')
axes[0].set_xlabel('Stage')
axes[0].set_ylim(0.0, 1.05)
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

axes[1].set_title('Stage Recovery Rate Within Input Subset')
axes[1].set_ylabel('Stage Recovery Rate')
axes[1].set_xlabel('Stage')
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[1].tick_params(axis='x', rotation=15)

plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for beam_size in beam_sizes:
    beam_frame = stage_df[stage_df['beam_size'] == beam_size].copy()
    axes[0].plot(beam_frame['stage_plot_label'], beam_frame['remaining_subset_size'], marker='o', label=f'beam={beam_size}')
    axes[1].plot(beam_frame['stage_plot_label'], beam_frame['suffix_all_zero_group_rate'], marker='o', label=f'beam={beam_size}')

axes[0].set_title('Remaining Hard-Set Size After Each Stage')
axes[0].set_ylabel('Remaining Sample Count')
axes[0].set_xlabel('Stage')
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

axes[1].set_title('All-Zero Rollout-Group Rate (Suffix-Only)')
axes[1].set_ylabel('All-Zero Group Rate')
axes[1].set_xlabel('Stage')
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[1].tick_params(axis='x', rotation=15)

plt.show()


## 前缀命中分布：`full` vs `suffix-only`

下面这两组堆叠柱状图回答的是：“所有 beam 候选加总起来看，命中长度分布在不同 stage 上是怎么变化的？”

读图时要注意：
- 颜色代表 `match_len`，也就是“连续命中的 token 数”，不是 reward 本身。
- `full` 视角会把提示 token 也算进来，所以在 `hint_k` 阶段天然会更乐观一些。
- `suffix-only` 先扣掉提示深度，更适合回答“给了脚手架之后，模型还能自己往后接多远”。
- 如果 `full` 看起来提升很大，但 `suffix-only` 提升很小，通常说明提升主要来自提示暴露了前缀，而不是续写能力真的改善了。


In [ ]:
def plot_histograms(hist_df: pd.DataFrame, title_prefix: str):
    fig, axes = plt.subplots(len(beam_sizes), 1, figsize=(14, 4 * len(beam_sizes)), constrained_layout=True)
    if len(beam_sizes) == 1:
        axes = [axes]

    for axis, beam_size in zip(axes, beam_sizes):
        subset = hist_df[hist_df['beam_size'] == beam_size].copy()
        pivot = subset.pivot_table(index='stage_plot_label', columns='match_len', values='share', fill_value=0.0)
        ordered_stage_labels = [stage_plot_label(stage) for stage in sorted(subset['stage_name'].unique(), key=stage_sort_key)]
        pivot = pivot.reindex(index=ordered_stage_labels)
        pivot.plot(kind='bar', stacked=True, ax=axis, colormap='viridis')
        axis.set_title(f'{title_prefix} Prefix-Match Distribution (beam={beam_size})')
        axis.set_ylabel('Share of Candidates')
        axis.set_xlabel('Stage')
        axis.set_ylim(0.0, 1.0)
        axis.grid(alpha=0.2, axis='y')
        axis.legend(title='Match Length', bbox_to_anchor=(1.02, 1), loc='upper left')
        axis.tick_params(axis='x', rotation=15)

    plt.show()


plot_histograms(full_hist_df, 'Full')
plot_histograms(suffix_hist_df, 'Suffix-Only')


## 典型 rollout-group 模式

这一部分回答的是：“把同一个样本的一组 beam 看成一个整体时，典型的失败/成功形状长什么样？”

可以把 pattern 当成一张压缩后的 beam 画像：
- `0:8` 或 `0:16`：所有 beam 都在第一个待预测位置就失败，是最硬的模式。
- `0:4|1:4`：有一半 beam 连第一个 token 都没对上，另一半只对了 1 个 token。
- `3:8`：8 条 beam 都已经对到长度 3，说明这个 stage 的主要难点已经被压缩到更靠后的 token 上。

如果 `full` 视角的 pattern 明显变好，但 `suffix-only` 仍然偏向 `0:*`，说明 hint 只是把前面的脚手架搭好了，真正困难的续写部分并没有被解决。


In [ ]:
def top_patterns(pattern_df: pd.DataFrame, beam_size: int, stage_name: str, top_k: int = 12) -> pd.DataFrame:
    subset = pattern_df[(pattern_df['beam_size'] == beam_size) & (pattern_df['stage_name'] == stage_name)].copy()
    return subset.sort_values('count', ascending=False).head(top_k)


def plot_top_patterns(pattern_df: pd.DataFrame, *, suffix: bool):
    stage_names = sorted(stage_df['stage_name'].unique(), key=stage_sort_key)
    fig, axes = plt.subplots(len(beam_sizes), len(stage_names), figsize=(4.8 * len(stage_names), 4.2 * len(beam_sizes)), constrained_layout=True)
    if len(beam_sizes) == 1:
        axes = [axes]

    for row_index, beam_size in enumerate(beam_sizes):
        for col_index, stage_name in enumerate(stage_names):
            axis = axes[row_index][col_index]
            top = top_patterns(pattern_df, beam_size, stage_name)
            if top.empty:
                axis.axis('off')
                continue
            axis.barh(top['pattern'][::-1], top['count'][::-1], color='#4C78A8')
            axis.set_title(f"beam={beam_size} · {stage_plot_label(stage_name)}")
            axis.set_xlabel('Sample Count')
            if col_index == 0:
                axis.set_ylabel('Pattern')
            axis.grid(alpha=0.2, axis='x')
    suffix_label = 'Suffix-Only' if suffix else 'Full'
    fig.suptitle(f'Top Rollout-Group Patterns ({suffix_label})', fontsize=16)
    plt.show()


plot_top_patterns(full_pattern_df, suffix=False)
plot_top_patterns(suffix_pattern_df, suffix=True)


## 汇总指标与自动结论

上面的图适合“看形状”，下面这两部分适合“抓结论”：

- 第一张表把每个 `beam_size` 的关键数字压缩到一行，方便横向比较。
- 第二段自动结论会把“主拐点在哪一层 hint”“最后还剩多少困难子集”“全零组有没有明显下降”直接写成中文摘要。


In [ ]:
def build_derived_metrics(stage_frame: pd.DataFrame, summary_dict: dict):
    rows = []
    for beam_size, payload in sorted(((int(k), v) for k, v in summary_dict['results'].items()), key=lambda x: x[0]):
        beam_frame = stage_frame[stage_frame['beam_size'] == beam_size].set_index('stage_name')
        cumulative = payload['cumulative']
        rows.append(
            {
                'beam_size': beam_size,
                'base_hit_rate': beam_frame.loc['base', 'cumulative_rule_hit_sample_rate'],
                'hint_1_increment': beam_frame.loc['hint_1', 'stage_rule_hit_sample_rate_within_input'] if 'hint_1' in beam_frame.index else None,
                'hint_2_increment': beam_frame.loc['hint_2', 'stage_rule_hit_sample_rate_within_input'] if 'hint_2' in beam_frame.index else None,
                'hint_3_increment': beam_frame.loc['hint_3', 'stage_rule_hit_sample_rate_within_input'] if 'hint_3' in beam_frame.index else None,
                'cumulative_after_hint_1': beam_frame.loc['hint_1', 'cumulative_rule_hit_sample_rate'] if 'hint_1' in beam_frame.index else None,
                'cumulative_after_hint_2': beam_frame.loc['hint_2', 'cumulative_rule_hit_sample_rate'] if 'hint_2' in beam_frame.index else None,
                'cumulative_after_hint_3': beam_frame.loc['hint_3', 'cumulative_rule_hit_sample_rate'] if 'hint_3' in beam_frame.index else None,
                'final_remaining_subset_size': cumulative['final_remaining_subset_size'],
                'final_remaining_subset_rate': cumulative['final_remaining_subset_rate'],
                'base_full_all_zero_rate': beam_frame.loc['base', 'full_all_zero_group_rate'],
                'hint_1_suffix_all_zero_rate': beam_frame.loc['hint_1', 'suffix_all_zero_group_rate'] if 'hint_1' in beam_frame.index else None,
                'hint_2_suffix_all_zero_rate': beam_frame.loc['hint_2', 'suffix_all_zero_group_rate'] if 'hint_2' in beam_frame.index else None,
                'hint_3_suffix_all_zero_rate': beam_frame.loc['hint_3', 'suffix_all_zero_group_rate'] if 'hint_3' in beam_frame.index else None,
            }
        )
    return pd.DataFrame(rows)


derived_df = build_derived_metrics(stage_df, summary).rename(
    columns={
        'beam_size': 'Beam 宽度',
        'base_hit_rate': 'base 命中率',
        'hint_1_increment': 'hint_1 追回率',
        'hint_2_increment': 'hint_2 追回率',
        'hint_3_increment': 'hint_3 追回率',
        'cumulative_after_hint_1': 'hint_1 后累计命中率',
        'cumulative_after_hint_2': 'hint_2 后累计命中率',
        'cumulative_after_hint_3': 'hint_3 后累计命中率',
        'final_remaining_subset_size': '最终剩余困难样本数',
        'final_remaining_subset_rate': '最终剩余困难样本占比',
        'base_full_all_zero_rate': 'base 的 Full 全零组占比',
        'hint_1_suffix_all_zero_rate': 'hint_1 的 Suffix 全零组占比',
        'hint_2_suffix_all_zero_rate': 'hint_2 的 Suffix 全零组占比',
        'hint_3_suffix_all_zero_rate': 'hint_3 的 Suffix 全零组占比',
    }
)

display(
    format_percent_columns(
        derived_df,
        [
            'base 命中率',
            'hint_1 追回率',
            'hint_2 追回率',
            'hint_3 追回率',
            'hint_1 后累计命中率',
            'hint_2 后累计命中率',
            'hint_3 后累计命中率',
            '最终剩余困难样本占比',
            'base 的 Full 全零组占比',
            'hint_1 的 Suffix 全零组占比',
            'hint_2 的 Suffix 全零组占比',
            'hint_3 的 Suffix 全零组占比',
        ],
    )
)


In [ ]:
def narrative_lines(summary_dict: dict, stage_frame: pd.DataFrame):
    lines = []
    for beam_size, payload in sorted(((int(k), v) for k, v in summary_dict['results'].items()), key=lambda x: x[0]):
        beam_frame = stage_frame[stage_frame['beam_size'] == beam_size].set_index('stage_name')
        cumulative = payload['cumulative']
        lines.append(f"### beam={beam_size}")
        lines.append(f"- `base`（无提示）的精确命中率是 **{beam_frame.loc['base', 'cumulative_rule_hit_sample_rate']:.2%}**。")
        if 'hint_1' in beam_frame.index:
            lines.append(
                f"- `hint_1` 在 `base` 尚未命中的样本里追回了 **{beam_frame.loc['hint_1', 'stage_rule_hit_sample_rate_within_input']:.2%}**，累计命中率提升到 **{beam_frame.loc['hint_1', 'cumulative_rule_hit_sample_rate']:.2%}**。"
            )
        if 'hint_2' in beam_frame.index:
            lines.append(
                f"- `hint_2` 是主要拐点：它进一步追回了 `hint_1` 剩余困难子集中的 **{beam_frame.loc['hint_2', 'stage_rule_hit_sample_rate_within_input']:.2%}**。"
            )
        if 'hint_3' in beam_frame.index:
            lines.append(
                f"- `hint_3` 更像上界分析 / 最后一跳诊断阶段；跑完后最终剩余困难样本占比是 **{cumulative['final_remaining_subset_rate']:.4%}**。"
            )
        lines.append(f"- `base` 的 Full 视角全零组占比是 **{beam_frame.loc['base', 'full_all_zero_group_rate']:.2%}**。")
        if 'hint_1' in beam_frame.index:
            lines.append(f"- `hint_1` 后的 Suffix-only 全零组占比是 **{beam_frame.loc['hint_1', 'suffix_all_zero_group_rate']:.2%}**。")
        if 'hint_2' in beam_frame.index:
            lines.append(f"- `hint_2` 后的 Suffix-only 全零组占比是 **{beam_frame.loc['hint_2', 'suffix_all_zero_group_rate']:.2%}**。")
        if 'hint_3' in beam_frame.index:
            lines.append(f"- `hint_3` 后的 Suffix-only 全零组占比是 **{beam_frame.loc['hint_3', 'suffix_all_zero_group_rate']:.2%}**。")
        lines.append('')
    return lines


display(Markdown("\n".join(narrative_lines(summary, stage_df))))


In [ ]:
if details is None:
    display(Markdown('## 最终残留困难样本导出\n\n**跳过导出：** 未找到 `details.json`，因此无法定位最后阶段仍未命中的具体样本。'))
else:
    residual_dir = Path('output/jupyter-notebook/genrec-hint-cascade-artifacts')
    residual_dir.mkdir(parents=True, exist_ok=True)
    export_frames = []

    for beam_size in beam_sizes:
        final_stage = summary['results'][str(beam_size)]['cumulative']['final_stage']
        stage_rows = details['results'][str(beam_size)]['stages'][final_stage]['rows']
        residual_rows = [row for row in stage_rows if not row['group']['rule_hit_any']]
        frame = pd.DataFrame(
            {
                'beam_size': beam_size,
                'final_stage': final_stage,
                'sample_id': [row['sample_id'] for row in residual_rows],
                'source_index': [row.get('source_index') for row in residual_rows],
                'hint_text': [row.get('hint_text', '') for row in residual_rows],
                'suffix_pattern': [row['group'].get('suffix_group_pattern', '') for row in residual_rows],
                'full_pattern': [row['group'].get('full_group_pattern', '') for row in residual_rows],
            }
        )
        out_path = residual_dir / f'beam{beam_size}_{final_stage}_residual.csv'
        frame.to_csv(out_path, index=False)
        export_frames.append((beam_size, final_stage, out_path, frame))

    display(Markdown('## 最终残留困难样本导出\n这些样本是在最后一个 cascade stage 之后仍未命中的顽固难例。这里展示的是预览，完整结果会写到 CSV。'))
    for beam_size, final_stage, out_path, frame in export_frames:
        print(f'beam={beam_size} 残留样本数: {len(frame)} -> {out_path}')
        display(Markdown(f"### beam={beam_size} · 最终阶段 `{final_stage}`\n- 残留样本数：`{len(frame)}`\n- 导出路径：`{out_path}`"))
        display(
            frame.rename(
                columns={
                    'beam_size': 'Beam 宽度',
                    'final_stage': '最终阶段',
                    'sample_id': '样本 ID',
                    'source_index': '原始索引',
                    'hint_text': '提示文本',
                    'suffix_pattern': 'Suffix 模式',
                    'full_pattern': 'Full 模式',
                }
            )
        )


## 下一步怎么用这份结果

- 优先检查“核心总表 + 两张折线图”，判断真正的收益主要来自 `hint_1`、`hint_2` 还是更深的 hint。
- 如果 `hint_2` 已经解决了绝大多数困难子集，而 `hint_3` 只剩极少数样本，通常可以把 `hint_3` 看成诊断上界，而不是默认策略。
- 重点比较 `full` 和 `suffix-only`：
  - 两者都明显改善，说明模型的续写能力确实在变好。
  - 只有 `full` 改善，而 `suffix-only` 没明显变化，说明提示更多是在“送前缀”。
- 最后结合导出的 residual CSV，看剩余样本是否集中在少数固定 pattern（例如一直是 `0:beam_size`），这通常意味着某些 token 组合或索引区段仍然存在系统性难点。
